In [ ]:
!pip install -q --upgrade scikit-learn

In [ ]:
!pip install skops

In [ ]:
!pip install -q datasets huggingface_hub skops scikit-learn numpy pandas


In [ ]:

import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from sklearn.metrics import roc_auc_score, accuracy_score
import json, pickle, os

In [ ]:
api = HfApi()

# List all models under KingTechnician that match the probe naming convention
all_models = api.list_models(author="KingTechnician", search="tiu-acts")
probe_repos = [
    m.id for m in all_models
    if m.id.endswith("-probe")
]
probe_repos.sort()

print(f"Found {len(probe_repos)} probe repos:")
for r in probe_repos:
    print(f"  {r}")

In [ ]:
def acts_dataset_from_probe(probe_repo: str) -> str:
    """
    'KingTechnician/tiu-acts-llama3-8b-instruct-layer-12-probe'
    -> 'KingTechnician/tiu-acts-llama3-8b-instruct-layer-12'
    """
    assert probe_repo.endswith("-probe")
    return probe_repo[: -len("-probe")]

In [ ]:
import skops.io as sio

def load_probe_and_score(probe_repo: str, acts: np.ndarray):
    # Download truth directions
    td_path = hf_hub_download(repo_id=probe_repo, filename="truth_directions.json")
    with open(td_path) as f:
        td = json.load(f)
    t_g = np.array(td["t_G"])
    t_p = np.array(td["t_P"]) if "t_P" in td and td["t_P"] is not None else None

    # Download the skops model
    model_path = hf_hub_download(repo_id=probe_repo, filename="model.skops")

    # First, inspect what types need to be trusted
    unknown_types = sio.get_untrusted_types(file=model_path)
    print(f"  Untrusted types: {unknown_types}")

    # Load with those types trusted
    lr = sio.load(model_path, trusted=unknown_types)

    # Project activations into 2D truth plane
    proj_tg = acts @ t_g
    if t_p is not None:
        t_p_arr = np.array(t_p).flatten()
        proj_tp = acts @ t_p_arr
        acts_2d = np.column_stack([proj_tg, proj_tp])
    else:
        acts_2d = proj_tg[:, None]

    scores = lr.predict_proba(acts_2d)[:, 1]
    return scores

In [ ]:
def sweep_threshold(scores: np.ndarray, labels: np.ndarray, n_steps=1000):
    """
    Sweep τ from 0 to 1, return (best_tau, best_acc, acc_at_0.5, auroc).
    """
    auroc = roc_auc_score(labels, scores)

    # Accuracy at default τ=0.5
    acc_default = accuracy_score(labels, (scores >= 0.5).astype(int))

    # Sweep
    best_tau, best_acc = 0.5, acc_default
    for tau in np.linspace(0, 1, n_steps + 1):
        preds = (scores >= tau).astype(int)
        acc = accuracy_score(labels, preds)
        if acc > best_acc:
            best_acc = acc
            best_tau = tau

    return best_tau, best_acc, acc_default, auroc

In [ ]:
results = []

for probe_repo in probe_repos:
    acts_repo = acts_dataset_from_probe(probe_repo)
    print(f"\n{'='*60}")
    print(f"Probe: {probe_repo}")
    print(f"Acts:  {acts_repo}")

    # Load activation dataset (test split)
    try:
        ds = load_dataset(acts_repo, split="test")
    except Exception as e:
        print(f"  SKIP — could not load dataset: {e}")
        continue

    # Find the activation column (e.g. "layer_12_acts")
    act_cols = [c for c in ds.column_names if c.startswith("layer_") and c.endswith("_acts")]
    if not act_cols:
        print(f"  SKIP — no activation column found. Columns: {ds.column_names}")
        continue
    act_col = act_cols[0]
    print(f"  Using activation column: {act_col}")

    # Extract activations and labels
    acts = np.array(ds.with_format("numpy")[act_col], dtype=np.float32)
    labels = np.array(ds.with_format("numpy")["label"], dtype=int)
    print(f"  Test samples: {len(labels)}, dims: {acts.shape[1]}")

    # Score
    try:
        scores = load_probe_and_score(probe_repo, acts)
    except Exception as e:
        print(f"  SKIP — probe loading failed: {e}")
        continue

    # Sweep
    best_tau, best_acc, acc_05, auroc = sweep_threshold(scores, labels)

    print(f"  Acc(τ=0.5):    {acc_05:.4f}")
    print(f"  Acc(τ*={best_tau:.3f}): {best_acc:.4f}")
    print(f"  AUROC:         {auroc:.4f}")

    results.append({
        "probe": probe_repo.split("/")[-1],
        "τ*": round(best_tau, 4),
        "Acc(τ*)": round(best_acc, 4),
        "Acc(τ=0.5)": round(acc_05, 4),
        "AUROC": round(auroc, 4),
        "Δ Acc": round(best_acc - acc_05, 4),
        "n_test": len(labels),
    })

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

df = pd.DataFrame(results)
df = df.sort_values("AUROC", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(16, 0.8 * len(df) + 2.5))
ax.axis("off")
ax.set_title("Threshold Calibration Diagnostic — All Probes", fontsize=14, fontweight="bold", pad=12)

# Shorten probe names for readability
display_df = df.copy()
display_df["probe"] = display_df["probe"].str.replace("tiu-acts-", "").str.replace("-probe", "")

col_labels = display_df.columns.tolist()
cell_text = display_df.values.tolist()

table = ax.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 2.0)

# Style header
for j in range(len(col_labels)):
    cell = table[0, j]
    cell.set_facecolor("#2d3436")
    cell.set_text_props(color="white", fontweight="bold")

# Color-code Δ Acc column
delta_col_idx = col_labels.index("Δ Acc")
for i in range(1, len(cell_text) + 1):
    val = float(cell_text[i - 1][delta_col_idx])
    if val > 0.01:
        table[i, delta_col_idx].set_facecolor("#ffeaa7")
    elif val > 0.05:
        table[i, delta_col_idx].set_facecolor("#fab1a0")

    # Alternate row shading
    if i % 2 == 0:
        for j in range(len(col_labels)):
            if table[i, j].get_facecolor() == (1.0, 1.0, 1.0, 1.0):
                table[i, j].set_facecolor("#f5f6fa")

table.auto_set_column_width(col=list(range(len(col_labels))))
plt.tight_layout()
plt.savefig("calibration_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary
flagged = df[df["Δ Acc"] > 0.01]
if len(flagged):
    print(f"\n{len(flagged)} probe(s) show >1pp accuracy gain from threshold recalibration.")
else:
    print("\nNo probes show substantial undercalibration (all Δ Acc ≤ 1pp).")